In [1]:
import sqlite3 as sql
import pandas as pd
# ==================== 模擬建立試算的資料數據 ====================
# 學員: 林其生 , 學號: DB00237
data = [
    {"city": "taipei", "customer_type": "new", "amount": 1200},
    {"city": "taipei", "customer_type": "new", "amount": 800},
    {"city": "taipei", "customer_type": "old", "amount": 1500},
    {"city": "taichung", "customer_type": "new", "amount": 900},
    {"city": "taichung", "customer_type": "new", "amount": 1100},
    {"city": "taichung", "customer_type": "old", "amount": 2000},
    {"city": "kaohsiung", "customer_type": "new", "amount": 700},
    {"city": "kaohsiung", "customer_type": "new", "amount": 1000},
    {"city": "kaohsiung", "customer_type": "old", "amount": 1800},
]
# ==================== 以sql 語法：建立資料表 ====================
connection = sql.connect(":memory:")

create_table_sql = '''
create table customer_orders (
    city ,
    customer_type ,
    amount 
)
'''

connection.execute(create_table_sql)
connection.executemany(
    "insert into customer_orders (city, customer_type, amount) values (?, ?, ?)",
    [(row["city"], row["customer_type"], row["amount"]) for row in data],
)
connection.commit()
# ==================== sql 語法：篩選、分組、計算平均 ====================
average_amount_sql = '''
select
    city,
    count(*) as new_customer_count,
    round(avg(amount), 2) as average_amount
from customer_orders
where customer_type = 'new'
group by city
order by city
'''

sql_result = pd.read_sql_query(average_amount_sql, connection)
print("sql 統計結果：")
print(sql_result)
sql_result.to_csv('sql_result.csv')

# ==================== pandas 語法：建立 dataframe ====================
orders = pd.DataFrame(data)
# ==================== pandas 語法：篩選新客戶 ====================
new_orders = orders[orders["customer_type"] == "new"]
# ==================== pandas 語法：分組、計算平均 ====================
pandas_result = (
    new_orders.groupby("city", as_index=False)
    .agg(new_customer_count=("amount", "count"),
        average_amount=("amount", "mean"),
    )
    .sort_values("city")
)

pandas_result["average_amount"] = pandas_result["average_amount"].round(2)

print("\npandas 統計結果：")
print(pandas_result)
pandas_result.to_csv('pandas_result.csv')
connection.close()

ModuleNotFoundError: No module named 'pandas'